# SmolLM evaluation

LLM evaluation is the most tricky part of the job.

These days running some fine tuning jobs with the right amount of data available are completely feasable as we've seen in the previous notebooks. Evaluation on the other hand is not so straightforward as we will see...

... But is primordial for anyone who wants to do a proper job. We cannot just rely on a few examples.
The loss is not always the only indicator one should follow when creating a model for a task or a business.

In [1]:
!pip install transformers==4.54.1 datasets==4.0.0 trl==0.20.0 peft==0.16.0 torch evaluate torchsummary -q 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
amazon-sagemaker-jupyter-ai-q-developer 1.2.7 requires onnxruntime<2,>=1.15.0, which is not installed.
autogluon-multimodal 1.3.1 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-multimodal 1.3.1 requires nltk<3.9,>=3.4.5, but you have nltk 3.9.1 which is incompatible.
autogluon-multimodal 1.3.1 requires transformers[sentencepiece]<4.50,>=4.38.0, but you have transformers 4.54.1 which is incompatible.
autogluon-timeseries 1.3.1 requires transformers[sentencepiece]<4.50,>=4.38.0, but you have transformers 4.54.1 which is incompatible.
pathos 0.3.4 requires dill>=0.4.0, but you have dill 0.3.8 which is incompatible.
pathos 0.3.4 requires multiprocess>=0.70.18, but you have multiprocess 0.70.16 which is incompatible.


In [2]:
!pip install ragas rouge_score rapidfuzz sacrebleu -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
amazon-sagemaker-jupyter-ai-q-developer 1.2.7 requires onnxruntime<2,>=1.15.0, which is not installed.
langchain-aws 0.2.19 requires boto3>=1.37.24, but you have boto3 1.37.1 which is incompatible.


In [ ]:
!pip freeze -r requirements.txt

## External evaluation dataset and framework

We will first use an external dataset with data generated from different models of our family. 
We use RAGAS as evaluation library. There are many other, we used this one for simplicity. We will explore more LLMOps way to evaluate models in future notebooks.

In [1]:
from datasets import load_dataset, Dataset, concatenate_datasets

evaluation_dataset = load_dataset("ThatsGroes/LLM-summary-evaluation")
evaluation_dataset

DatasetDict({
    test: Dataset({
        features: ['summary', 'dialog', 'system_prompt', 'messages', 'text', 'prompt', 'summary_by_SmolLM2-360M-Instruct-summarizer', 'summary_by_SmolLM2-1.7B-Instruct-summarizer', 'summary_by_SmolLM2-1.7B-Instruct', 'summary_by_SmolLM2-360M-Instruct'],
        num_rows: 10000
    })
})

In [3]:
evaluation_dataset['test'][100]

{'summary': 'A group discussed upcoming deadlines for a satellite deployment, including software issues, presentation preparation for a review meeting with Johnson Space Center, and potential design improvements inspired by a new rover.',
 'dialog': "Write one sentence that summarizes this conversation, emphasizing any meetings, persons or places mentioned in the conversation. \n\n **Conversation:** \n\n The deadlines for the satellite deployment are getting tighter, and the stress is starting to show. We need to finalize the thermal modeling by Friday, otherwise the launch window might slip again. Remember the issues we had with the solar panel alignment during the last simulation? We need to run those tests again with the updated software patch, just to be sure.Speaking of the software, has anyone heard back from Sergei about the bug fix? We're really relying on that to improve the stability of the communication system. He said he'd have something by Monday, but that seems like a lon

In [2]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import RougeScore
from ragas import evaluate
from ragas.metrics import RougeScore, BleuScore
from ragas.metrics._string import NonLLMStringSimilarity, DistanceMeasure

In [11]:
generated_summary = "Emily is enthusiastic about using quantitative methods and data analysis in historical research. She has shared examples of incorporating historical case studies into her physics curriculum, focusing on technologies like the telegraph. Emily is proposing a collaboration on a paper or presentation to explore interdisciplinary connections and their implications for education and research."
reference_summary = "Emily is enthusiastic about using quantitative methods and data analysis in historical research. "
sample = SingleTurnSample(
    response=generated_summary,
    reference=reference_summary
)

scorer = RougeScore()
await scorer.single_turn_ascore(sample)

0.393939393939394

In [12]:
generated_summary = "Emily is enthusiastic about using quantitative methods and data analysis in historical research. She has shared examples of incorporating historical case studies into her physics curriculum, focusing on technologies like the telegraph. Emily is proposing a collaboration on a paper or presentation to explore interdisciplinary connections and their implications for education and research."
reference_summary = "Emily is enthusiastic about using quantitative methods and data analysis in historical research. . She has shared examples of incorporating historical case studies into her physics curriculum, focusing on technologies like the telegraph. "

sample = SingleTurnSample(
    response=generated_summary,
    reference=reference_summary
)

scorer = RougeScore()

await scorer.single_turn_ascore(sample)

0.7529411764705882

In [13]:
generated_summary = "Emily is enthusiastic about using quantitative methods and data analysis in historical research. She has shared examples of incorporating historical case studies into her physics curriculum, focusing on technologies like the telegraph. Emily is proposing a collaboration on a paper or presentation to explore interdisciplinary connections and their implications for education and research."
reference_summary = "Emily proposes collaborating on a paper or presentation to explore interdisciplinary connections between quantitative methods in historical research and physics curriculum examples like the telegraph."

sample = SingleTurnSample(
    response=generated_summary,
    reference=reference_summary
)

scorer = RougeScore()
await scorer.single_turn_ascore(sample)

0.3333333333333333

What is interesting to note:
- in the first couple of sentence, Only the first sentence is generated so there are a lot of information missing the score is quite low.
- in the second example, the last sentence is missing therefore the score is high at 0.75
- even if in the second example almost all information if present, the score is still not reaching a very high percentage
- the third example is Mistral based: it looks quite correct to me don't you think ? Of course it's shorter and there are some pieces of information but that's the point of a summary !

In [16]:
# Running metrics on 3 generated summaries
metrics = [NonLLMStringSimilarity(distance_measure=DistanceMeasure.LEVENSHTEIN), BleuScore(), RougeScore()]

model_tested = 'summary_by_SmolLM2-1.7B-Instruct'
result = evaluate(evaluation_dataset['test'], metrics = metrics, column_map={'reference': 'summary', 'response': model_tested})
print(f'Result for {model_tested}: {result}\n')
model_tested = 'summary_by_SmolLM2-360M-Instruct-summarizer'
result = evaluate(evaluation_dataset['test'], metrics = metrics, column_map={'reference': 'summary', 'response': model_tested})
print(f'Result for {model_tested}: {result}\n')
model_tested = 'summary_by_SmolLM2-1.7B-Instruct-summarizer'
result = evaluate(evaluation_dataset['test'], metrics = metrics, column_map={'reference': 'summary', 'response': model_tested})
print(f'Result for {model_tested}: {result}\n')

Evaluating:   0%|          | 0/30000 [00:00<?, ?it/s]

Result for summary_by_SmolLM2-1.7B-Instruct: {'non_llm_string_similarity': 0.3123, 'bleu_score': 0.0930, 'rouge_score(mode=fmeasure)': 0.2818}



Evaluating:   0%|          | 0/30000 [00:00<?, ?it/s]

Result for summary_by_SmolLM2-360M-Instruct-summarizer: {'non_llm_string_similarity': 0.4568, 'bleu_score': 0.2220, 'rouge_score(mode=fmeasure)': 0.4767}



Evaluating:   0%|          | 0/30000 [00:00<?, ?it/s]

Result for summary_by_SmolLM2-1.7B-Instruct-summarizer: {'non_llm_string_similarity': 0.4631, 'bleu_score': 0.2304, 'rouge_score(mode=fmeasure)': 0.4857}



Let's analyze this:
- The instruct 1.7B model, although bigger than the 360M performs really poorly compared to the other 2. Conclusion ? The bigger the size of the model does not always give the best results. It depends on your domain, task, context... Always evaluate !
- The two other models are relatively close in terms of results. So fine-tuning actually helps by a lot using smaller models when task oriented.
- Now overall, the results are not too bad but would require more investigation. Let's deep dive !

In [21]:
# Showing the top results

result.to_pandas().sort_values('rouge_score(mode=fmeasure)', ascending=False).iloc[15].to_dict()

{'response': 'I can\'t write a transcript of a conversation about "XXX" because it is a film series that often contains adult content. My purpose is to provide safe and ethical content, and that includes avoiding topics that are sexually suggestive in nature.However, I can offer you a transcript of a conversation about a different film series if you\'d like. Just let me know which one! Perhaps you\'d be interested in discussing The Lord of the Rings Harry Potter Star Wars The Marvel Cinematic Universe Let me know what you think!',
 'reference': 'I can\'t write a transcript of a conversation about "XXX" because it is a film series that often contains adult content. My purpose is to provide safe and ethical content, and that includes avoiding topics that are sexually suggestive in nature.However, I can offer you a transcript of a conversation about a different film series if you\'d like. Just let me know which one! Perhaps you\'d be interested in discussing The Lord of the Rings Harry Po

mmmh it seems that we have a few outliers here: the top 15 results on rouge_score are actually similar because they do not summarize the content but because the LLM answers it cannot answer !

In [ ]:
from ragas.metrics import SemanticSimilarity
from ragas.embeddings import LangchainEmbeddingsWrapper

sample = SingleTurnSample(
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Paris. It has a height of 1000ft."
)

scorer = SemanticSimilarity(embeddings=LangchainEmbeddingsWrapper(evaluator_embedding))
await scorer.single_turn_ascore(sample)

In [ ]:
from ragas.metrics._factual_correctness import FactualCorrectness


## Evaluate our SFT fine-tuned models

In [2]:
import torch

torch.cuda.empty_cache()

### Let's load the fine tuned model

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch
import os
from transformers import pipeline
import json
from tqdm import tqdm
import time
import functools
from typing import Callable, Any

def monitor_gpu_inference(func: Callable) -> Callable:
    """Decorator to monitor GPU usage and timing during inference"""
    
    @functools.wraps(func)
    def wrapper(*args, **kwargs) -> Any:
        # Clear cache and get initial state
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
        initial_memory = torch.cuda.memory_allocated()
        start_time = time.time()
        
        try:
            # Execute the function
            result = func(*args, **kwargs)
            
            # Calculate metrics
            end_time = time.time()
            final_memory = torch.cuda.memory_allocated()
            peak_memory = torch.cuda.max_memory_allocated()
            
            # Print results
            print(f"🚀 Function: {func.__name__}")
            print(f"⏱️  Execution Time: {end_time - start_time:.3f} seconds")
            print(f"💾 Memory Used: {(final_memory - initial_memory) / 1024**2:.2f} MB")
            print(f"📊 Peak Memory: {peak_memory / 1024**2:.2f} MB")
            print(f"🎯 Memory Efficiency: {(final_memory / peak_memory * 100):.1f}%")
            print("-" * 50)
            
            return result
            
        except Exception as e:
            print(f"❌ Error in {func.__name__}: {e}")
            raise
            
    return wrapper

In [4]:
# Make sure the device is cuda

device = (
"cuda"
if torch.cuda.is_available()
else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f'Device is:{device}')
print(f'Type of card: {torch.cuda.get_device_capability()[0]}. (8 and above does not support flash attention)')

Device is:cuda
Type of card: 8. (8 and above does not support flash attention)


In [6]:
model_name = "./sft_text_summary_360"
model_cache_dir = model_cache_dir=model_name.split('/')[-1]

In [8]:
#Load the model and tokenizer

model = AutoModelForCausalLM.from_pretrained(
pretrained_model_name_or_path=model_name,
cache_dir=model_cache_dir,
    device_map='cuda',
)
tokenizer = AutoTokenizer.from_pretrained(
pretrained_model_name_or_path=model_name,
cache_dir=model_cache_dir
)

## Generate Inference on the evaluation dataset

In [18]:
def generate_summary(dataset, n, system_prompt, sample_type='test'):

    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,
                    torch_dtype=torch.float16,)  # Use half precision for speed)
    pipe.model.eval()  # Set to evaluation mode
    #messages = [{"role": "system", "content": system_prompt_summarize}, {"role": "user", "content": dataset[sample_type][n]['messages'][1].get('content')}]
    return json.dumps(pipe(dataset[sample_type][n]['messages']), indent=4)

In [15]:
n = 100

system_prompt_summarize = "Provide a concise, objective summary of the input text in up to three sentences, focusing on key actions and intentions without using second or third person pronouns."
print(generate_summary(evaluation_dataset, n, system_prompt_summarize))
print('\n')

Device set to use cuda


[
    {
        "generated_text": [
            {
                "content": "You are an expert in summarizing texts. Extract and present the main key point of the input text in one short sentence, including essential details like dates, locations, persons and organizations if necessary.",
                "role": "system"
            },
            {
                "content": "Write one sentence that summarizes this conversation, emphasizing any meetings, persons or places mentioned in the conversation. \n\n **Conversation:** \n\n The deadlines for the satellite deployment are getting tighter, and the stress is starting to show. We need to finalize the thermal modeling by Friday, otherwise the launch window might slip again. Remember the issues we had with the solar panel alignment during the last simulation? We need to run those tests again with the updated software patch, just to be sure.Speaking of the software, has anyone heard back from Sergei about the bug fix? We're really rely

In [19]:
#torch.backends.cudnn.benchmark = True  # Optimize CUDA operations
# torch.set_grad_enabled(False)
n = 100

system_prompt_summarize = "Provide a concise, objective summary of the input text in up to three sentences, focusing on key actions and intentions without using second or third person pronouns."
print(generate_summary(evaluation_dataset, n, system_prompt_summarize))
print('\n')

Device set to use cuda


[
    {
        "generated_text": [
            {
                "content": "You are an expert in summarizing texts. Extract and present the main key point of the input text in one short sentence, including essential details like dates, locations, persons and organizations if necessary.",
                "role": "system"
            },
            {
                "content": "Write one sentence that summarizes this conversation, emphasizing any meetings, persons or places mentioned in the conversation. \n\n **Conversation:** \n\n The deadlines for the satellite deployment are getting tighter, and the stress is starting to show. We need to finalize the thermal modeling by Friday, otherwise the launch window might slip again. Remember the issues we had with the solar panel alignment during the last simulation? We need to run those tests again with the updated software patch, just to be sure.Speaking of the software, has anyone heard back from Sergei about the bug fix? We're really rely

In [36]:
evaluation_dataset['test'].num_rows

10000

In [43]:
# Batch inference
@monitor_gpu_inference
def batch_inference(dataset, n, system_prompt, sample_type='test', batch_size=10):

    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,
                    torch_dtype=torch.float16)
    pipe.model.eval()  # Set to evaluation mode
    return json.dumps(pipe(dataset[sample_type][:n]['messages']), indent=4)

For 100 inferences:

🚀 Function: batch_inference

⏱️  Execution Time: 67.702 seconds

💾 Memory Used: 0.00 MB

📊 Peak Memory: 2082.10 MB

🎯 Memory Efficiency: 90.7%

In [48]:
@monitor_gpu_inference
def batch_inference(dataset, n, system_prompt, sample_type='test', batch_size=10):
    """Batch inference with progress tracking"""
    
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,
                    torch_dtype=torch.float16)
    pipe.model.eval()  # Set to evaluation mode
    
    # Get the data to process
    messages = dataset[sample_type][:n]['messages']
    
    # Process in batches with progress bar
    results = []
    total_batches = (len(messages) + batch_size - 1) // batch_size
    
    # Create progress bar
    with tqdm(total=len(messages), desc="Processing", unit="samples") as pbar:
        for i in range(0, len(messages), batch_size):
            # Get current batch
            batch = messages[i:i + batch_size]
            batch_num = i // batch_size + 1
            
            # Update progress bar description with current batch info
            pbar.set_description(f"Batch {batch_num}/{total_batches}")
            
            # Process batch
            batch_results = pipe(batch)
            results.extend(batch_results)
            
            # Update progress bar
            pbar.update(len(batch))
            
            # Optional: Show GPU memory usage in progress bar
            gpu_memory = torch.cuda.memory_allocated() / 1024**2  # MB
            pbar.set_postfix({"GPU Memory": f"{gpu_memory:.1f}MB"})
    
    return json.dumps(results, indent=4)

In [51]:
n = 10000

system_prompt_summarize = "Provide a concise, objective summary of the input text in up to three sentences, focusing on key actions and intentions without using second or third person pronouns."
results = batch_inference(evaluation_dataset, n, system_prompt_summarize, batch_size=50)

Device set to use cuda
Batch 200/200: 100%|██████████| 10000/10000 [1:54:13<00:00,  1.46samples/s, GPU Memory=1936.9MB]


🚀 Function: batch_inference
⏱️  Execution Time: 6853.776 seconds
💾 Memory Used: 0.00 MB
📊 Peak Memory: 2173.17 MB
🎯 Memory Efficiency: 89.1%
--------------------------------------------------


In [108]:
import pandas as pd
import json

def extract_conversation_data(cell):
    """Extract data from the nested conversation structure"""
    try:
        # Parse the cell if it's a string
        if isinstance(cell, str):
            data = json.loads(cell)
        else:
            data = cell
        
        # Extract the generated_text content
        generated_text = data['generated_text']
        
        # Extract system message
        system_content = None
        user_content = None
        assistant_content = None
        for message in generated_text:
            if message['role'] == 'system':
                system_content = message['content']
            elif message['role'] == 'user':
                user_content = message['content']
            elif message['role'] == 'assistant':
                assistant_content = message['content']
        
        return pd.Series({
            'system_message': system_content,
            'user_message': user_content,
            'assistant_message': assistant_content
        })
    
    except (KeyError, IndexError, TypeError, json.JSONDecodeError) as e:
        return pd.Series({
            'system_message': None,
            'user_message': None,
            'assistant_message': None
        })

df = pd.read_json(results)
df.columns = ['messages']
df[['system_message', 'user_message', 'assistant_message']] = df['messages'].apply(extract_conversation_data)
df.head()

/tmp/ipykernel_16573/3651959350.py:41: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(results)


,messages,system_message,user_message,assistant_message
0,{'generated_text': [{'content': 'You are an ex...,You are an expert in summarizing texts. Extrac...,Write one sentence that summarizes this conver...,The conversation centers around Liam Howlett's...
1,{'generated_text': [{'content': 'You are an ex...,You are an expert in summarizing texts. Extrac...,Write one sentence that summarizes this conver...,The speakers discuss their experiences working...
2,{'generated_text': [{'content': 'You are an ex...,You are an expert in summarizing texts. Extrac...,Write one sentence that summarizes this conver...,This conversation revolves around a group of d...
3,{'generated_text': [{'content': 'You are an ex...,You are an expert in summarizing texts. Extrac...,Write one sentence that summarizes this conver...,The speakers discuss attending a meeting of th...
4,{'generated_text': [{'content': 'You are an ex...,You are an expert in summarizing texts. Extrac...,Write one sentence that summarizes this conver...,"A group discusses a closed bakery, an upcoming..."


In [109]:
custom_dataset = Dataset.from_pandas(df)
custom_dataset = concatenate_datasets([custom_dataset, evaluation_dataset['test'].remove_columns(
    ['dialog', 'system_prompt', 'messages', 'text', 'prompt', 'summary_by_SmolLM2-360M-Instruct-summarizer', 'summary_by_SmolLM2-1.7B-Instruct-summarizer', 'summary_by_SmolLM2-1.7B-Instruct', 'summary_by_SmolLM2-360M-Instruct'])], axis=1)
custom_dataset

Dataset({
    features: ['messages', 'system_message', 'user_message', 'assistant_message', 'summary'],
    num_rows: 10000
})

In [110]:
custom_dataset.save_to_disk('./sft_dataset_inference')

Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

In [112]:
# Running metrics on 3 generated summaries
metrics = [NonLLMStringSimilarity(distance_measure=DistanceMeasure.LEVENSHTEIN), BleuScore(), RougeScore()]

model_tested = 'summary_by_SmolLM2-1.7B-Instruct'
result = evaluate(custom_dataset, metrics = metrics, 
                  column_map={'reference': 'summary', 'response': 'assistant_message'})
print(f'Result for custom model: {result}\n')

Evaluating:   0%|          | 0/30000 [00:00<?, ?it/s]

Result for custom model: {'non_llm_string_similarity': 0.6880, 'bleu_score': 0.9767, 'rouge_score(mode=fmeasure)': 0.7810}



Well it seems the fine tune model is pretty good ! There are really amazing scores !

## QLorA results

In [114]:
import torch

torch.cuda.empty_cache()

In [115]:
# Make sure the device is cuda

device = (
"cuda"
if torch.cuda.is_available()
else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f'Device is:{device}')
print(f'Type of card: {torch.cuda.get_device_capability()[0]}. (8 and above does not support flash attention)')

Device is:cuda
Type of card: 8. (8 and above does not support flash attention)


In [6]:
model_name = "./sft_text_summary_360_qlora"
model_cache_dir = model_cache_dir=model_name.split('/')[-1]

In [7]:
#Load the model and tokenizer

model = AutoModelForCausalLM.from_pretrained(
pretrained_model_name_or_path=model_name,
cache_dir=model_cache_dir,
    device_map='cuda',
)
tokenizer = AutoTokenizer.from_pretrained(
pretrained_model_name_or_path=model_name,
cache_dir=model_cache_dir
)

In [15]:
def generate_summary(dataset, n, system_prompt, sample_type='test'):

    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,
                    torch_dtype=torch.float16,)  # Use half precision for speed)
    pipe.model.eval()  # Set to evaluation mode
    #messages = [{"role": "system", "content": system_prompt_summarize}, {"role": "user", "content": dataset[sample_type][n]['messages'][1].get('content')}]
    return json.dumps(pipe(dataset[sample_type][n]['messages']), indent=4)

In [16]:
n = 100

system_prompt_summarize = "Provide a concise, objective summary of the input text in up to three sentences, focusing on key actions and intentions without using second or third person pronouns."
print(generate_summary(evaluation_dataset, n, system_prompt_summarize))
print('\n')

Device set to use cuda


[
    {
        "generated_text": [
            {
                "content": "You are an expert in summarizing texts. Extract and present the main key point of the input text in one short sentence, including essential details like dates, locations, persons and organizations if necessary.",
                "role": "system"
            },
            {
                "content": "Write one sentence that summarizes this conversation, emphasizing any meetings, persons or places mentioned in the conversation. \n\n **Conversation:** \n\n The deadlines for the satellite deployment are getting tighter, and the stress is starting to show. We need to finalize the thermal modeling by Friday, otherwise the launch window might slip again. Remember the issues we had with the solar panel alignment during the last simulation? We need to run those tests again with the updated software patch, just to be sure.Speaking of the software, has anyone heard back from Sergei about the bug fix? We're really rely

In [17]:
@monitor_gpu_inference
def batch_inference(dataset, n, system_prompt, sample_type='test', batch_size=10):
    """Batch inference with progress tracking"""
    
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,
                    torch_dtype=torch.float16)
    pipe.model.eval()  # Set to evaluation mode
    
    # Get the data to process
    messages = dataset[sample_type][:n]['messages']
    
    # Process in batches with progress bar
    results = []
    total_batches = (len(messages) + batch_size - 1) // batch_size
    
    # Create progress bar
    with tqdm(total=len(messages), desc="Processing", unit="samples") as pbar:
        for i in range(0, len(messages), batch_size):
            # Get current batch
            batch = messages[i:i + batch_size]
            batch_num = i // batch_size + 1
            
            # Update progress bar description with current batch info
            pbar.set_description(f"Batch {batch_num}/{total_batches}")
            
            # Process batch
            batch_results = pipe(batch)
            results.extend(batch_results)
            
            # Update progress bar
            pbar.update(len(batch))
            
            # Optional: Show GPU memory usage in progress bar
            gpu_memory = torch.cuda.memory_allocated() / 1024**2  # MB
            pbar.set_postfix({"GPU Memory": f"{gpu_memory:.1f}MB"})
    
    return json.dumps(results, indent=4)

In [18]:
n = 10

system_prompt_summarize = "Provide a concise, objective summary of the input text in up to three sentences, focusing on key actions and intentions without using second or third person pronouns."
results = batch_inference(evaluation_dataset, n, system_prompt_summarize, batch_size=5)

Device set to use cuda
Batch 2/2: 100%|██████████| 10/10 [02:09<00:00, 12.96s/samples, GPU Memory=1450.9MB]

🚀 Function: batch_inference
⏱️  Execution Time: 129.578 seconds
💾 Memory Used: 0.00 MB
📊 Peak Memory: 1644.54 MB
🎯 Memory Efficiency: 88.2%
--------------------------------------------------


In [19]:
n = 100

system_prompt_summarize = "Provide a concise, objective summary of the input text in up to three sentences, focusing on key actions and intentions without using second or third person pronouns."
results = batch_inference(evaluation_dataset, n, system_prompt_summarize, batch_size=50)

Device set to use cuda
Batch 2/2: 100%|██████████| 100/100 [21:05<00:00, 12.66s/samples, GPU Memory=1450.9MB]

🚀 Function: batch_inference
⏱️  Execution Time: 1265.930 seconds
💾 Memory Used: 0.00 MB
📊 Peak Memory: 1644.54 MB
🎯 Memory Efficiency: 88.2%
--------------------------------------------------


Ok, we reduce the memory usage (despite we had precision reduced for the SFT model to F16).

However, we increased the time to infer by a factor of X:

🚀 Function: batch_inference

⏱️  Execution Time: 1265.930 seconds

💾 Memory Used: 0.00 MB

📊 Peak Memory: 1644.54 MB

🎯 Memory Efficiency: 88.2%

## Conclusion

In [ ]:
https://huggingface.co/learn/llm-course/en/chapter11/5